# Detection — Training & Validation

**Model:** YOLOv8m, 2 classes (`person`, `ball`)  
**Dataset:** 971 frames, 16 matches (822 train / 149 val)  
**Metrics:** mAP50=0.944, mAP50-95=0.748, Precision=0.952, Recall=0.892  
**Weights:** `models/detection/weights/best.pt`

> **When to use this notebook:** Only when adding new labeled data and retraining.  
> For per-game processing, start from `01_team_classification.ipynb`.

In [ ]:
import sys
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import importlib
from pathlib import Path
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import src.config, src.detection, src.video_utils, src.visualization, src.dataset
for mod in [src.config, src.detection, src.video_utils, src.visualization, src.dataset]:
    importlib.reload(mod)

from src.config import Config
from src.detection import PlayerDetector
from src.video_utils import open_video
from src.visualization import Annotator
from src.dataset import create_data_yaml, split_val

print('Loaded.')

## Dataset

In [ ]:
dataset_dir = Config.PROJECT_ROOT / 'data' / 'object_detection'
all_imgs   = list((dataset_dir / 'images' / 'all').glob('*.jpg'))
train_imgs = list((dataset_dir / 'images' / 'train').glob('*.jpg'))
val_imgs   = list((dataset_dir / 'images' / 'val').glob('*.jpg')) if (dataset_dir / 'images' / 'val').exists() else []

print(f'Dataset: {dataset_dir}')
print(f'  All:   {len(all_imgs)} frames')
print(f'  Train: {len(train_imgs)} frames')
print(f'  Val:   {len(val_imgs)} frames')

if torch.cuda.is_available():
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    device_arch = f'sm_{cc_major}{cc_minor}'
    supported   = set(torch.cuda.get_arch_list())
    gpu_name    = torch.cuda.get_device_name(0)
    arch_status = 'OK' if device_arch in supported else 'MISSING — install a newer PyTorch build (CUDA 12.8+)'
    print(f'CUDA: {gpu_name} ({device_arch}) — arch {arch_status}')
else:
    print('CUDA: not available')

## Training

> **Only run this cell when you have new labeled data.**  
> Current model at `models/detection/weights/best.pt` was trained on 971 frames.

In [ ]:
dataset_dir = Config.PROJECT_ROOT / 'data' / 'object_detection'

create_data_yaml(dataset_dir)
split_val(dataset_dir)

for cache in (dataset_dir / 'labels').rglob('*.cache'):
    cache.unlink()
    print(f'Deleted stale cache: {cache.name}')

torch.backends.cudnn.benchmark = True
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

use_cuda = False
if torch.cuda.is_available():
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    if f'sm_{cc_major}{cc_minor}' in set(torch.cuda.get_arch_list()):
        use_cuda = True

device  = 0 if use_cuda else 'cpu'
workers = max(2, min(8, os.cpu_count() or 2))
batch   = 8 if use_cuda else 2  # batch=8 is well-utilised on 12GB VRAM with yolov8m + imgsz=1280

device_label = 'CUDA' if use_cuda else 'CPU'
print(f'Device: {device_label}')

# yolov8n=3M params  yolov8s=11M  yolov8m=25M  yolov8l=43M
MODEL = 'yolov8m.pt'

model = YOLO(MODEL)
model.train(
    data          = str(dataset_dir / 'data.yaml'),
    epochs        = 150,
    imgsz         = 1280,
    batch         = batch,
    device        = device,
    workers       = workers,
    mosaic        = 1.0,
    copy_paste    = 0.3,
    cache         = True,
    amp           = use_cuda,
    deterministic = False,
    name          = 'football_2class_yolov8m',
    patience      = 30,
)

## Error Analysis

Run the trained model on **val** images and display:
- **Class confusions** — GT says person but model says ball (or vice-versa)
- **False positives** — high-confidence detections with no matching GT box
- **False negatives** — GT boxes the model completely missed

In [ ]:
IOU_THRESHOLD   = 0.5    # IoU to count as a match
CONF_THRESHOLD  = 0.25   # prediction confidence cutoff
PAD             = 15     # pixels of padding around crops
MAX_SHOW        = 30     # max crops per category
CLASSES         = {0: 'person', 1: 'ball'}

# Filter: None = show all, 0 = person only, 1 = ball only
FP_CLASS_FILTER = 1
FN_CLASS_FILTER = 1

weights = Config.resolve_yolo_model()
print(f'Model: {weights}')
model = YOLO(weights)

dataset_dir  = Config.PROJECT_ROOT / 'data' / 'object_detection'
val_imgs_dir = dataset_dir / 'images' / 'val'
val_lbls_dir = dataset_dir / 'labels' / 'val'
val_images   = sorted(val_imgs_dir.glob('*.jpg'))
print(f'Val images: {len(val_images)}')


def yolo_to_xyxy(cx, cy, w, h, img_w, img_h):
    return [(cx - w/2)*img_w, (cy - h/2)*img_h, (cx + w/2)*img_w, (cy + h/2)*img_h]


def iou(a, b):
    x1 = max(a[0], b[0]);  y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]);  y2 = min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-6)


def pad_crop(img, box):
    h, w = img.shape[:2]
    return img[max(0, int(box[1])-PAD):min(h, int(box[3])+PAD),
               max(0, int(box[0])-PAD):min(w, int(box[2])+PAD)]


confusions, false_pos, false_neg = [], [], []

for img_path in val_images:
    img     = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ih, iw  = img.shape[:2]

    lbl_path = val_lbls_dir / img_path.with_suffix('.txt').name
    gt_boxes = []
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            gt_boxes.append((int(parts[0]), yolo_to_xyxy(*map(float, parts[1:5]), iw, ih)))

    pred_boxes = []
    for r in model.predict(img_path, conf=CONF_THRESHOLD, verbose=False):
        for box in r.boxes:
            pred_boxes.append((int(box.cls[0]), box.xyxy[0].cpu().numpy().tolist(), float(box.conf[0])))

    gt_matched   = [False] * len(gt_boxes)
    pred_matched = [False] * len(pred_boxes)
    pairs = sorted(
        [(iou(pb, gb), pi, gi) for pi, (pc, pb, _) in enumerate(pred_boxes)
                               for gi, (gc, gb) in enumerate(gt_boxes)
                               if iou(pb, gb) >= IOU_THRESHOLD],
        reverse=True,
    )

    for _, pi, gi in pairs:
        if pred_matched[pi] or gt_matched[gi]:
            continue
        pred_matched[pi] = gt_matched[gi] = True
        pc, pb, pconf = pred_boxes[pi]
        gc, _         = gt_boxes[gi]
        if pc != gc:
            confusions.append((pad_crop(img_rgb, pb), gc, pc, pconf, img_path.name))

    for pi, matched in enumerate(pred_matched):
        if not matched:
            pc, pb, pconf = pred_boxes[pi]
            false_pos.append((pad_crop(img_rgb, pb), pc, pconf, img_path.name))

    for gi, matched in enumerate(gt_matched):
        if not matched:
            gc, gb = gt_boxes[gi]
            false_neg.append((pad_crop(img_rgb, gb), gc, img_path.name))

fp_filtered = sorted(
    [x for x in false_pos if FP_CLASS_FILTER is None or x[1] == FP_CLASS_FILTER],
    key=lambda x: -x[2],
)
fn_filtered = [x for x in false_neg if FN_CLASS_FILTER is None or x[1] == FN_CLASS_FILTER]

print(f'Confusions:      {len(confusions)}')
print(f'False positives: {len(false_pos)} total, {len(fp_filtered)} after class filter')
print(f'False negatives: {len(false_neg)} total, {len(fn_filtered)} after class filter')


def show_crops(items, title, label_fn):
    if not items:
        print(f'{title}: none')
        return
    n    = min(len(items), MAX_SHOW)
    cols = min(8, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(2.5*cols, 3*rows))
    fig.suptitle(f'{title}  ({len(items)} total, showing {n})', fontsize=13)
    axes = np.atleast_2d(axes).flatten()
    for i in range(n):
        c = items[i][0]
        if c.size == 0:
            axes[i].axis('off')
            continue
        axes[i].imshow(c)
        axes[i].set_title(label_fn(items[i]), fontsize=7)
        axes[i].axis('off')
    for j in range(n, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()


show_crops(confusions, 'Class Confusions',
           lambda x: f'GT={CLASSES[x[1]]} Pred={CLASSES[x[2]]} {x[3]:.2f}')
show_crops(fp_filtered, f'False Positives ({CLASSES.get(FP_CLASS_FILTER, "all")})',
           lambda x: f'{CLASSES[x[1]]} conf={x[2]:.2f}')
show_crops(fn_filtered, f'False Negatives ({CLASSES.get(FN_CLASS_FILTER, "all")})',
           lambda x: f'missed {CLASSES[x[1]]}')

## Detection Preview

Sanity-check the current model on a single gameplay frame.

In [ ]:
PREVIEW_SLUG   = 'sut-mla'
PREVIEW_OFFSET = 25 * 60  # seconds into the match

detector = PlayerDetector()
cap      = open_video(Config.MATCH_VIDEOS[PREVIEW_SLUG])
fps      = cap.get(cv2.CAP_PROP_FPS)
cap.set(cv2.CAP_PROP_POS_FRAMES, int(PREVIEW_OFFSET * fps))
ret, frame = cap.read()
cap.release()

detections = detector.detect(frame)
n_players  = len(detections['players'])
n_ball     = len(detections['ball'])
print(f'{PREVIEW_SLUG} @ {PREVIEW_OFFSET//60}:{PREVIEW_OFFSET%60:02d} — players: {n_players}, ball: {n_ball}')

annotator   = Annotator()
debug_frame = annotator.annotate_frame_debug(frame, detections)

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(debug_frame, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f'{PREVIEW_SLUG} — {n_players} players, {n_ball} ball')
plt.tight_layout()
plt.show()